# Process real-world annotation results

This notebook has two functions: filter the formal annotations and create the real-world flood-depth distribution figure used in the paper. It outputs one CSV file, one JSON file, and one PDF figure.

## 1. Filter annotations and create ground truth

Only formal annotations (`phrase == "real"`) corresponding to images in the real-world evaluation dataset are used. In the paper study, each image was independently annotated by three participants. Each relative annotation is first converted to an absolute depth in centimetres using the height of the selected vehicle. If the difference between the maximum and minimum of the three absolute depths is no greater than 10 cm, all three annotations are retained. Otherwise, the annotation with the largest deviation from the mean of the other two is removed. The retained absolute depths are averaged at full numerical precision, and only the final image-level ground truth is rounded for output.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import json
import math
import os
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        app_dir = candidate / 'scripts' / 'build_real_world_dataset' / 'annotation_app'
        if app_dir.is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the FloodDepth-Retrieval repository.')


REPO_ROOT = find_repo_root()
DEFAULT_DB_PATH = (
    REPO_ROOT / 'scripts' / 'build_real_world_dataset' /
    'annotation_app' / 'results' / 'results.db'
)
DB_PATH = Path(os.environ.get('FLOOD_APP_DB_PATH', DEFAULT_DB_PATH)).expanduser().resolve()
EVALUATION_IMAGES_DIR = REPO_ROOT / 'data' / 'evaluation_dataset' / 'real_world' / 'images'
OUTPUT_DIR = DB_PATH.parent / 'processed'
OUTLIER_THRESHOLD_CM = 10.0
REFERENCE_VEHICLE_HEIGHT_CM = 145.7

CAR_HEIGHT_CM = {
    '2014_volkswagen_golf_gti_mk7_Compact': 146.9,
    '2016_bmw_x5_m_SUV': 175.5,
    '2021_bmw_218i_gran_coupe_15_Family': 142.0,
    '2021_volkswagen_cross_touran_l_280_tsi_MPV': 167.4,
    '2022_peugeot_308_Hachback': 145.7,
    '2024_byd_sealion_7_Estate': 162.0,
    'mini_cooper_s_pre-facelift_City': 142.7,
    'tesla_model_3_Compact': 144.1,
    '2022_Peugeot_308': 145.7,
}

print(f'Database: {DB_PATH}')
print(f'Evaluation images: {EVALUATION_IMAGES_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
def basename(value):
    return Path(urlparse(str(value)).path).name


def model_name(value):
    return Path(urlparse(str(value)).path).stem


def keep_indices(depths_cm, threshold_cm=10.0):
    depths_cm = np.asarray(depths_cm, dtype=float)
    if len(depths_cm) != 3 or depths_cm.max() - depths_cm.min() <= threshold_cm:
        return np.arange(len(depths_cm))

    deviations = []
    for index, value in enumerate(depths_cm):
        deviations.append(abs(value - np.delete(depths_cm, index).mean()))
    outlier_index = int(np.argmax(deviations))
    return np.setdiff1d(np.arange(3), [outlier_index])


def map_to_level(depth_cm, reference_height_cm=145.7, intervals=10):
    if depth_cm <= 0:
        return 0
    return min(math.ceil(depth_cm / (reference_height_cm / intervals)), intervals - 1)


if not DB_PATH.is_file():
    raise FileNotFoundError(f'Annotation database not found: {DB_PATH}')
if not EVALUATION_IMAGES_DIR.is_dir():
    raise FileNotFoundError(f'Evaluation image directory not found: {EVALUATION_IMAGES_DIR}')

evaluation_image_names = {
    path.name for path in EVALUATION_IMAGES_DIR.iterdir() if path.is_file()
}

with sqlite3.connect(DB_PATH) as connection:
    annotations = pd.read_sql_query(
        '''
        SELECT id, user, image_path, car_model, flood_depth,
               timestamp, phrase, consent_date
        FROM results
        WHERE phrase = 'real'
        ORDER BY id
        ''',
        connection,
    )

annotations['image_name'] = annotations['image_path'].map(basename)
annotations = annotations[annotations['image_name'].isin(evaluation_image_names)].copy()
annotations['car_model'] = annotations['car_model'].map(model_name)
annotations['flood_depth'] = annotations['flood_depth'].astype(float)

invalid_ratios = annotations[~annotations['flood_depth'].between(0.0, 1.0)]
if not invalid_ratios.empty:
    raise ValueError(f"Invalid flood_depth in database rows: {invalid_ratios['id'].tolist()}")

annotations['car_height_cm'] = annotations['car_model'].map(CAR_HEIGHT_CM)
unknown_models = sorted(annotations.loc[annotations['car_height_cm'].isna(), 'car_model'].unique())
if unknown_models:
    raise ValueError(f'Missing vehicle heights for: {unknown_models}')

# Preserve full floating-point precision through filtering and aggregation.
annotations['abs_depth_cm'] = annotations['flood_depth'] * annotations['car_height_cm']
filtered_groups = []
for _, group in annotations.groupby('image_path', sort=True):
    positions = keep_indices(group['abs_depth_cm'].to_numpy(), OUTLIER_THRESHOLD_CM)
    filtered_groups.append(group.iloc[positions])
filtered_annotations = pd.concat(filtered_groups, ignore_index=True)

aggregated = (
    filtered_annotations.groupby('image_path', as_index=False)
    .agg(abs=('abs_depth_cm', 'mean'))
)
# Match the source-code order: compute ratio from the unrounded mean, then round abs.
aggregated['ratio'] = (aggregated['abs'] / REFERENCE_VEHICLE_HEIGHT_CM).round(2)
aggregated['abs'] = aggregated['abs'].round(2)
aggregated['flood_level'] = aggregated['abs'].map(
    lambda depth: map_to_level(float(depth), REFERENCE_VEHICLE_HEIGHT_CM)
)
aggregated['image_name'] = aggregated['image_path'].map(basename)
ground_truth = (
    aggregated.set_index('image_name')[['abs', 'ratio', 'flood_level']]
    .sort_index()
    .to_dict(orient='index')
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / 'filtered_annotations.csv'
csv_fields = [
    'id', 'user', 'image_path', 'image_name', 'car_model',
    'car_height_cm', 'flood_depth', 'abs_depth_cm',
    'timestamp', 'phrase', 'consent_date',
]
csv_output = filtered_annotations[csv_fields].copy()
csv_output['abs_depth_cm'] = csv_output['abs_depth_cm'].round(2)
csv_output.to_csv(csv_path, index=False)

json_path = OUTPUT_DIR / 'real_world_gt_info.json'
with json_path.open('w', encoding='utf-8') as handle:
    json.dump(ground_truth, handle, indent=4, ensure_ascii=False)
    handle.write('\n')

print(f'Formal annotations: {len(annotations)}')
print(f'Outliers removed: {len(annotations) - len(filtered_annotations)}')
print(f'Ground-truth images: {len(ground_truth)}')
print(f'Saved: {csv_path}')
print(f'Saved: {json_path}')

## 2. Create the flood-depth distribution figure

The final image-level depths are grouped into the same 10 cm intervals used in the paper.

In [ ]:
depths_cm = [entry['abs'] for entry in ground_truth.values()]
bin_edges = list(range(10, 121, 10))
bin_centers = [(left + right) / 2 for left, right in zip(bin_edges[:-1], bin_edges[1:])]
bin_labels = [f'{left}–{right}' for left, right in zip(bin_edges[:-1], bin_edges[1:])]

fig, ax = plt.subplots(figsize=(12, 5))
counts, _, patches = ax.hist(
    depths_cm, bins=bin_edges, color='#0072B2', alpha=0.8,
    edgecolor='black', linewidth=1.0,
)
for count, patch in zip(counts, patches):
    if count > 0:
        ax.text(
            patch.get_x() + patch.get_width() / 2, count + 1, str(int(count)),
            ha='center', va='bottom', fontsize=20,
        )

ax.set_xticks(bin_centers, bin_labels, fontsize=16, rotation=45)
ax.set_xlabel('Flood depth (cm)', fontsize=20)
ax.set_ylabel('Image count', fontsize=20)
ax.set_yticks([])
ax.tick_params(axis='both', length=0)
ax.set_ylim(0, max(counts, default=0) + 5)
fig.tight_layout()

figure_path = OUTPUT_DIR / 'real-world_level_frequency.pdf'
fig.savefig(figure_path, bbox_inches='tight')
plt.show()
print(f'Saved: {figure_path}')